**Bronze Layer**

In [0]:
from pyspark import pipelines as dp

@dp.table(
    name="bronze_transactions",
    comment="Raw transaction data ingested from source table"
)
def bronze_transactions():
    """
    Bronze Layer: Ingest raw transaction data from the source streaming table.
    No transformation applied, just raw ingestion.
    """
    return spark.readStream.table("dataengineering.project.transactions")

**Silver Layer**

In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

@dp.table(
    name="silver_transactions",
    comment="Cleaned transaction data with data quality checks applied",
    cluster_by=["transaction_date", "customer_id"]
)
@dp.expect_all_or_drop({
    "valid_transaction_id": "transaction_id IS NOT NULL",
    "valid_transaction_date": "transaction_date IS NOT NULL",
    "valid_customer_id": "customer_id IS NOT NULL",
    "valid_quantity": "quantity > 0",
    "valid_unit_price": "unit_price >= 0",
    "valid_total_amount": "total_amount >= 0"
})
@dp.expect("valid_product_info", "product_id IS NOT NULL AND product_name IS NOT NULL")
@dp.expect("valid_category", "category IS NOT NULL")
def silver_transactions():
    """
    Silver Layer: Clean and validate transaction data.
    - Applies data quality checks
    - Removes records with critical data quality issues
    - Warns on optional field issues
    - Drops _rescued_data column (used for Auto Loader error handling)
    """
    return (
        spark.readStream.table("bronze_transactions")
        .filter("_rescued_data IS NULL")  # Keep only clean records
        .drop("_rescued_data")
        .withColumn("transaction_date", F.col("transaction_date").cast("timestamp"))
        .withColumn("product_name", F.trim(F.col("product_name")))
    )

**Gold Layer**

In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

@dp.materialized_view(
    name="gold_daily_transactions",
    comment="Daily aggregated transaction metrics for reporting and analytics",
    cluster_by=["transaction_day"]
)
def gold_daily_transactions():
    """
    Gold Layer: Daily transaction aggregations.
    Provides business metrics aggregated by day:
    - Total daily revenue
    - Transaction count
    - Average transaction amount
    - Unique customers
    - Total items sold
    """
    return (
        spark.read.table("silver_transactions")
        .withColumn("transaction_day", F.to_date("transaction_date"))
        .groupBy("transaction_day")
        .agg(
            F.sum("total_amount").alias("total_revenue"),
            F.count("transaction_id").alias("transaction_count"),
            F.avg("total_amount").alias("avg_transaction_amount"),
            F.countDistinct("customer_id").alias("unique_customers"),
            F.sum("quantity").alias("total_items_sold"),
            F.min("transaction_date").alias("first_transaction_time"),
            F.max("transaction_date").alias("last_transaction_time")
        )
        .orderBy("transaction_day")
    )
